##Parte 1: Importar dados e bibliotecas


In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel('Dataset_Hospital_ETL_50mil.xls')
print(df.columns)
df

Index(['id_atendimento', 'data_atendimento', 'paciente', 'cidade', 'hospital',
       'especialidade', 'procedimento', 'valor_procedimento', 'quantidade',
       'convenio'],
      dtype='object')


,id_atendimento,data_atendimento,paciente,cidade,hospital,especialidade,procedimento,valor_procedimento,quantidade,convenio
0,ATD000001,25/04/2025,Paciente 000001,taguatinga,Hospital Vida,Cardiologia,Fisioterapia,"R$ 666,64",3.0,SUS
1,ATD000002,12/08/2025,Paciente 000002,GUARÁ,Hospital Vida,Neurologia,Consulta,"930,82",5.0,SaúdeMais
2,ATD000003,09/03/2026,Paciente 000003,taguatinga,Hospital Central,Dermatologia,Ressonância,"3.446,37",2.0,SUS
3,ATD000004,16/04/2026,Paciente 000004,Gama,Hospital Regional,Cardiologia,Consulta,"1.351,11",4.0,VidaPlena
4,ATD000005,23/03/2025,Paciente 000005,gama,Hospital São Lucas,Pediatria,Tomografia,"2.064,94",1.0,SUS
...,...,...,...,...,...,...,...,...,...,...
49995,ATD049996,10/04/2026,Paciente 049996,Guará,Hospital Vida,Ortopedia,Ressonância,"R$ 1.557,53",3.0,VidaPlena
49996,ATD049997,26/03/2025,Paciente 049997,Taguatinga,Hospital Brasília,Pediatria,Eletrocardiograma,"2.092,93",2.0,VidaPlena
49997,ATD049998,31/08/2025,Paciente 049998,guará,Hospital Brasília,Oftalmologia,Eletrocardiograma,"1.763,70",5.0,VidaPlena
49998,ATD000219,27/06/2025,Paciente 000219,sobradinho,Hospital São Lucas,Clínica Geral,Eletrocardiograma,"R$ 878,52",4.0,UniSaúde


##Parte 2: Limpar dados

###2.1: Remover linhas vazias

In [3]:
print(df.isna().sum())
df = df.dropna(how='all')
print(df.isna().sum())


id_atendimento        250
data_atendimento      250
paciente              250
cidade                250
hospital              250
especialidade         250
procedimento          250
valor_procedimento    250
quantidade            250
convenio              250
dtype: int64
id_atendimento        0
data_atendimento      0
paciente              0
cidade                0
hospital              0
especialidade         0
procedimento          0
valor_procedimento    0
quantidade            0
convenio              0
dtype: int64


###2.2: Padronizar cidades

In [4]:
print(df['cidade'].unique())
df.loc[:,'cidade'] = df['cidade'].astype(str)
df.loc[:,'cidade'] = df['cidade'].str.strip()
df.loc[:,'cidade'] = df['cidade'].str.lower().str.title()
print(df['cidade'].unique())

['taguatinga ' 'GUARÁ' 'taguatinga' ' Gama ' 'gama' 'águas claras '
 'SOBRADINHO' 'GAMA' ' Sobradinho ' 'sobradinho' ' Ceilândia '
 'Águas Claras' 'ÁGUAS CLARAS' 'águas claras' 'brasília' 'Taguatinga'
 'Gama' 'samambaia ' 'samambaia' 'guará ' 'ceilândia' 'sobradinho '
 'Brasília' ' Águas Claras ' 'SAMAMBAIA' ' Guará ' 'ceilândia '
 ' Samambaia ' 'gama ' 'guará' 'Sobradinho' 'Guará' 'Samambaia' 'BRASÍLIA'
 'CEILÂNDIA' 'TAGUATINGA' ' Taguatinga ' 'Ceilândia' ' Brasília '
 'brasília ']
['Taguatinga' 'Guará' 'Gama' 'Águas Claras' 'Sobradinho' 'Ceilândia'
 'Brasília' 'Samambaia']


###2.3: Padronizar valores unitários

In [5]:
print(df['valor_procedimento'].unique())
df.loc[:,'valor_procedimento'] = df['valor_procedimento'].astype(str)
df.loc[:,'valor_procedimento'] = (df['valor_procedimento']
                                  .str.replace('R$','', regex=False)
                                  .str.replace('.','', regex=False)
                                  .str.replace(',','.', regex=False)
                                  )
df.loc[:,'valor_procedimento'] = pd.to_numeric(df['valor_procedimento'], errors='coerce')
print(df['valor_procedimento'].unique())

['R$ 666,64' '930,82' '3.446,37' ... '2.092,93' '1.763,70' '1.553,10']
[666.64 930.82 3446.37 ... 2092.93 1763.7 1553.1]


###2.4: Remover datas inválidas

In [7]:
print(df['data_atendimento'].head(10))
df.loc[:, 'data_atendimento'] = pd.to_datetime(df['data_atendimento'],errors='coerce',dayfirst=True)
print('Datas inválidas:',df['data_atendimento'].isna().sum())

0    25/04/2025
1    12/08/2025
2    09/03/2026
3    16/04/2026
4    23/03/2025
5    25/06/2025
6    02/10/2025
7    29/07/2026
8    06/06/2025
9    01/10/2025
Name: data_atendimento, dtype: object
Datas inválidas: 742


###2.5: Remover dados duplicados

In [9]:
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.duplicated().sum())

498
0


###2.6: Remover valores negativos

In [10]:
print('Valores de quantidade negativa antes da correção:')
display(df[df['quantidade'] < 0]['quantidade'])

df.loc[df["quantidade"] < 0, "quantidade"] = 0

print('\nValores de quantidade negativa depois da correção (deve estar vazio):')
display(df[df['quantidade'] < 0]['quantidade'])

Valores de quantidade negativa antes da correção:


,quantidade
22,-3.0
26,-1.0
32,-3.0
92,-1.0
101,-1.0
...,...
49726,-1.0
49739,-2.0
49753,-2.0
49908,-2.0



Valores de quantidade negativa depois da correção (deve estar vazio):


,quantidade


##3: Salvar os dados limpos

In [12]:
print("\nFormato final:", df.head)
print(df.describe)


Formato final: <bound method NDFrame.head of       id_atendimento     data_atendimento         paciente      cidade  \
0          ATD000001  2025-04-25 00:00:00  Paciente 000001  Taguatinga   
1          ATD000002  2025-08-12 00:00:00  Paciente 000002       Guará   
2          ATD000003  2026-03-09 00:00:00  Paciente 000003  Taguatinga   
3          ATD000004  2026-04-16 00:00:00  Paciente 000004        Gama   
4          ATD000005  2025-03-23 00:00:00  Paciente 000005        Gama   
...              ...                  ...              ...         ...   
49994      ATD049995  2026-01-26 00:00:00  Paciente 049995       Guará   
49995      ATD049996  2026-04-10 00:00:00  Paciente 049996       Guará   
49996      ATD049997  2025-03-26 00:00:00  Paciente 049997  Taguatinga   
49997      ATD049998  2025-08-31 00:00:00  Paciente 049998       Guará   
49999      ATD050000  2025-08-06 00:00:00  Paciente 050000        Gama   

                   hospital  especialidade       procedimento  \
